# Random Forests

Random forests are a parallel ensemble method that uses a number of decision trees to make predictions. The trees are grown in a random order, and the final prediction is the average of the predictions of all the trees. This method is very fast and easy to implement, but it is prone to overfitting.

## Core Components

- Bootstrapping
- Feature Sampling
- Wisdom of the Crowd

> **Diversity** + **Randomness** + **Wisdom of the Crowd** = Random Forests

In the decision trees used in this ensemble method we do not use any pruning, we want models with high variance and low bias to average the errors out and produce a model that is as accurate as possible.

## Key Hyperparameters

- **n_estimators:** Number of trees in the forest
- **max_features:** Number of features to consider when looking for the best split
- **max_depth:** Maximum depth of the tree
- **min_samples_split:** Minimum number of samples required to split an internal node
- **min_samples_leaf:** Minimum number of samples required to be at a leaf node
- **bootstrap:** Whether bootstrap samples are used when building trees

## Advantages:

- Easy to implement
- Reduces variance
- Can be used for classification and regression
- Can be used for feature selection
- Works well with large datasets

## Disadvantages:

- Computationally expensive
- Can be unstable
- Can be prone to overfitting
- Needs space to store the trees, maybe not suitable for real-time applications

> **Note:** Always consider the trade-off between accuracy and computational cost when using ensemble methods.

## Example of Random Forests

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, root_mean_squared_error, mean_absolute_error, r2_score, make_scorer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [2]:
data = fetch_california_housing()

X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Number of samples: {X.shape[0]}")
print(f"Number of features: {X.shape[1]}")

Number of samples: 20640
Number of features: 8


In [3]:
df = pd.DataFrame(
    data=X_train,
    columns=data['feature_names'],
)
df[data.target_names[0]] = y_train
df

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,3.2596,33.0,5.017657,1.006421,2300.0,3.691814,32.71,-117.03,1.030
1,3.8125,49.0,4.473545,1.041005,1314.0,1.738095,33.77,-118.16,3.821
2,4.1563,4.0,5.645833,0.985119,915.0,2.723214,34.66,-120.48,1.726
3,1.9425,36.0,4.002817,1.033803,1418.0,3.994366,32.69,-117.11,0.934
4,3.5542,43.0,6.268421,1.134211,874.0,2.300000,36.78,-119.80,0.965
...,...,...,...,...,...,...,...,...,...
16507,6.3700,35.0,6.129032,0.926267,658.0,3.032258,33.78,-117.96,2.292
16508,3.0500,33.0,6.868597,1.269488,1753.0,3.904232,34.02,-117.43,0.978
16509,2.9344,36.0,3.986717,1.079696,1756.0,3.332068,34.03,-118.38,2.221
16510,5.7192,15.0,6.395349,1.067979,1777.0,3.178891,37.58,-121.96,2.835


In [4]:
print(data.DESCR)


.. _california_housing_dataset:

California Housing dataset
--------------------------

**Data Set Characteristics:**

:Number of Instances: 20640

:Number of Attributes: 8 numeric, predictive attributes and the target

:Attribute Information:
    - MedInc        median income in block group
    - HouseAge      median house age in block group
    - AveRooms      average number of rooms per household
    - AveBedrms     average number of bedrooms per household
    - Population    block group population
    - AveOccup      average number of household members
    - Latitude      block group latitude
    - Longitude     block group longitude

:Missing Attribute Values: None

This dataset was obtained from the StatLib:
https://lib.stat.cmu.edu/datasets/houses.zip

The target variable is the median house value for California districts,
expressed in hundreds of thousands of dollars ($100,000).

This dataset was derived from the 1990 U.S. census, using one row per census
block group. A block g

In [5]:
def evaluate_random_forest(
		X_train: np.ndarray,
		X_train_scaled: np.ndarray,
		y_train: np.ndarray
) -> None:
	"""
	Evaluates the Random Forest Regressor on the training data.

	:param X_train: The training data.
	:param X_train_scaled: The scaled training data.
	:param y_train: The training labels.
	"""
	rf_reg = RandomForestRegressor(
		n_estimators=100,
		oob_score=True,
		n_jobs=-1,
		random_state=42
	)

	dataset_names = ["X_train", "X_train_scaled"]
	for i, x in [(0, X_train), (1, X_train_scaled)]:
		cv_scores = cross_val_score(
			estimator=rf_reg,
			X=x,
			y=y_train,
			scoring='r2',
			cv=5,
			n_jobs=-1
		)
		print(f"Model Trained on {dataset_names[i]}")
		print(f"CV Score: {cv_scores.mean()} +/- {cv_scores.std()}")

evaluate_random_forest(
	X_train=X_train,
	X_train_scaled=X_train_scaled,
	y_train=y_train
)

Model Trained on X_train
CV Score: 0.8044053691692102 +/- 0.005381263688186624
Model Trained on X_train_scaled
CV Score: 0.8044490027318245 +/- 0.005250708686881523


In [ ]:
rf_reg = RandomForestRegressor(
    oob_score=True,
    random_state=42,
    n_jobs=-1
)
param_grid = {
    'n_estimators': [500, 800, 1000],
    # 'max_depth': [10, 20, 50],
    # 'min_samples_split': [2, 3, 4, 5, 6],
    # 'min_samples_leaf': [1, 2, 3, 4, 5],
    # 'max_features': ['auto', 'sqrt', 'log2'],
    # 'criterion': ["squared_error", "absolute_error", "friedman_mse", "Poisson"]
}

grid = GridSearchCV(
    estimator=rf_reg,
    param_grid=param_grid,
    scoring=make_scorer(r2_score),
    n_jobs=-1,
    cv=5
)

grid.fit(X_train, y_train)

In [ ]:
grid.best_params_

{'n_estimators': 500}

In [ ]:
pipe = Pipeline(
	steps=[
		("scaler", StandardScaler()),
		("regressor", RandomForestRegressor(
            n_estimators=100,
		))
	]
)

